![Banner](../images/09_03_banner_rs_rader.png)


# RADARSAT Constellation Mission (RCM) Processing

This walks through processing an RCM scene from the raw product to analysis-ready layers, explaining what each step does and why.


## Overview

This walks through processing an RCM scene from the raw product to analysis-ready layers, explaining what each step does and why.


## Core concepts

### Learning objectives

1. Identify SLC, GRC, GRD, and geocoded RCM products, and the polarisation, before choosing a workflow.
2. Sketch the backscatter, compact-pol, and InSAR chains from reading the product through to a map-ready layer.
3. Calibrate detected GRD with the product LUT and noise XML, and keep sigma0, beta0, and gamma0 distinct.
4. Form the RH/RV covariance, Stokes parameters, and an m-χ decomposition, and terrain-correct those outputs afterwards.
5. Run noise removal, calibration, speckle filtering, terrain flattening, and Range-Doppler terrain correction, and do the statistics in linear units.
6. Coregister an SLC pair, form an interferogram, remove topographic phase, unwrap, and convert phase to line-of-sight displacement.
7. Call geolattice.rsense.sar.radarsat for compact-pol, GRD calibration, and Stripmap SLC annotation, and open product.xml so the calibration and geometry are available.

### Contents

1. Know the product before processing
2. Overall workflows
3. RADARSAT processing functions in GeoLattice
4. Additional Resources


## Know the product before processing

**RCM basics**
- C-band SAR at 5.405 GHz, so the wavelength λ is about 5.55 cm.
- Three satellites, each on a 12-day repeat. Together they give a 4-day exact-repeat orbit.
- Right-looking, with ascending and descending passes.

**Product levels**

| Product | Content | Geometry | Use |
|---|---|---|---|
| **SLC** | Complex (I/Q), amplitude + phase | Slant range | InSAR, polarimetry, CP decomposition |
| **GRC** | Complex, multilooked | Ground range | Polarimetry with ground spacing |
| **GRD** | Detected amplitude | Ground range (ellipsoid) | Backscatter mapping, classification |
| **GCC / GCD** | Complex / detected | Geocoded to ellipsoid (no DEM) | Quick look; still needs terrain correction |

**Polarization options**
- Single (HH or VV).
- Dual (HH+HV or VV+VH).
- Compact-pol (CP: transmit right-circular, receive H and V, giving the channels RH and RV).
- Quad-pol, available only in certain modes.

**Folder contents**
- `metadata/product.xml` holds the orbit state vectors, timing, incidence angles, beam, and the geolocation grid.
- `imagery/*.tif` has one GeoTIFF per channel.
- `calibration/` holds the lookup tables for σ⁰, β⁰, and γ⁰ (sigma, beta, and gamma nought) and the noise levels.

RCM orbits are precise and already in the metadata. Unlike Sentinel-1, no separate "Apply Orbit File" step is needed.


## Overall workflows

**Backscatter workflow (GRD):**
```
Read product.xml → Subset AOI → Thermal noise removal → Radiometric calibration
→ (Multilook) → Speckle filter → Terrain flattening → Terrain correction (RTC)
→ Linear→dB → Masking/QC → Stack & analyse
```

**Compact-pol workflow (SLC/GRC):**
```
Read → Calibrate (complex) → Build 2×2 covariance [C2] → Boxcar/Refined-Lee polarimetric filter
→ Stokes parameters → m-χ / m-δ decomposition (or pseudo-quad reconstruction)
→ Terrain correction → Classification
```

**InSAR workflow (SLC pairs or stacks):**
```
Coregister (DEM-assisted) → Interferogram + coherence → Topographic phase removal
→ Goldstein filter → Phase unwrapping (SNAPHU) → Phase→displacement → Geocode
→ Time series (SBAS/PS, e.g. MintPy)
```


## RADARSAT processing functions in GeoLattice

The mission-specific API is in `geolattice.rsense.sar.radarsat` (source: [`radarsat.py`](https://github.com/zia207/GeoLattice/blob/main/src/rsense/sar/radarsat.py)). It currently focuses on RCM compact-polarimetry, RCM Level-1 GRD radiometric calibration, and reading metadata for Stripmap SLC interferometry. Use the existing sensor-generic SAR/InSAR tools for shared operations such as multilooking, speckle filtering, SLC coregistration/resampling, and interferogram formation. These helpers do not perform a full end-to-end processor, DEM-based terrain correction, phase unwrapping, or time-series inversion.

### RCM compact-polarimetry

| Function | Purpose and result |
|---|---|
| `split_rrrl(rrrl)` | Splits an RCM-ARD `rrrl` two-band asset into its real and imaginary arrays (bands 0 and 1). |
| `rr_rl_to_stokes(rr, rl, rrrl_re, rrrl_im)` | Converts the RCM-ARD circular receive channels and complex cross term into Stokes parameters `s0`-`s3` and degree of polarization. |
| `rrrl_to_rhrv(rr, rl, rrrl_re, rrrl_im)` | Converts RCM-ARD circular-basis products into the equivalent RH/RV 2×2 covariance components `c11`, `c22`, `c12_re`, and `c12_im`. The output can be used with compatible dual-pol covariance tools. |
| `rhrv_to_stokes(c11, c22, c12_re, c12_im)` | Computes Stokes parameters and degree of polarization from an RH/RV covariance matrix. |
| `compact_pol_mchi(s0, s1, s2, s3)` | Computes RCM-convention m-χ surface (`ps`), double-bounce (`pd`), and volume (`pv`) powers from Stokes inputs. The documented RCM sign convention matters when interpreting classes. |
| `simulate_compact_pol(s_hh, s_hv, s_vh, s_vv, size=7)` | Simulates right-circular-transmit, linear-receive compact-pol data from RADARSAT-2 (or other) quad-pol complex channels. It averages the resulting covariance over the requested window; this is a simulation, not a native RADARSAT-2 compact-pol acquisition. |
| `gamma0_to_sigma0(gamma0, ratio)` | Converts RCM-ARD gamma0 to sigma0 using the product's per-pixel `gamma_to_sigma_ratio` layer. |

### RCM Level-1 GRD radiometric calibration

| Function | Purpose and result |
|---|---|
| `read_rcm_lut(xml_path)` | Parses a `lutSigma`, `lutBeta`, or `lutGamma` calibration XML file, returning the range-axis sample positions, offset, and gain values. |
| `interpolate_rcm_lut(lut, width)` | Interpolates the sparse range-only gain values to one gain per image column. |
| `read_rcm_noise_levels(xml_path, cal_type="Sigma Nought")` | Reads the selected calibration type's per-beam NESZ segments from the RCM noise XML. Choose the type matching the LUT being used. |
| `interpolate_rcm_noise(segments, width)` | Interpolates and joins those per-beam noise segments over the image columns. |
| `calibrate_rcm(dn, gain, offset=0.0, noise_db=None, floor=1e-6)` | Converts detected DN to calibrated linear backscatter using `(DN² + offset) / gain`, optionally subtracts the matching NESZ in linear units, sets zero DN to missing, and applies a small nonnegative floor. Select sigma0, beta0, or gamma0 LUT consistently with the intended output. |

Use the calibration LUT and matching noise calibration from the same product and polarization. The mission-specific module handles LUT/noise interpretation; generic functions in `geolattice.rsense.sar.sentinel1` can then be used for operations such as multilooking and dB conversion. Terrain flattening and DEM-based terrain correction require an appropriate external workflow.

### RCM Stripmap SLC annotation for InSAR

`read_rcm_annotation(xml_path)` extracts the timing, range sampling, wavelength, pixel spacing, incidence limits, and orbit state vectors from an RCM Level-1 Stripmap SLC `product.xml`. It returns an annotation dictionary shaped for the sensor-generic Sentinel-1 `back_geocode` and `resample_slc` routines. It supports a single continuous Stripmap image, not ScanSAR burst/sub-swath geometry. RCM's precise state vectors are included in the product metadata, so this workflow does not require a separate orbit-file download.

### Example call patterns

```python
from geolattice.rsense.sar import radarsat

# RCM-ARD compact-pol assets (rrrl is the real/imaginary two-band asset)
rrrl_re, rrrl_im = radarsat.split_rrrl(rrrl)
stokes = radarsat.rr_rl_to_stokes(rr, rl, rrrl_re, rrrl_im)
scattering = radarsat.compact_pol_mchi(
    stokes["s0"], stokes["s1"], stokes["s2"], stokes["s3"]
)

# Level-1 GRD: use the matching calibration LUT and noise XML
lut = radarsat.read_rcm_lut(lut_xml)
gain = radarsat.interpolate_rcm_lut(lut, width=dn.shape[-1])
noise = radarsat.read_rcm_noise_levels(noise_xml, cal_type="Sigma Nought")
noise_db = radarsat.interpolate_rcm_noise(noise, width=dn.shape[-1])
sigma0 = radarsat.calibrate_rcm(
    dn, gain, offset=lut["offset"], noise_db=noise_db
)

# Stripmap SLC: supply the parsed annotation to shared coregistration tools
annotation = radarsat.read_rcm_annotation(product_xml)
```

The names `rr`, `rl`, `rrrl`, `dn`, and the XML paths above stand for arrays/assets and metadata belonging to the selected product. Check the product's polarization basis, calibration type, pixel units, supported beam mode, and function docstrings before applying these examples.


### Backscatter workflow, step by step

**Step 1, Read and subset**
- Open `product.xml` in SNAP, not the TIFF. The TIFF alone has no radar geometry.
- Subset to your AOI immediately. RCM scenes are large, and every later step scales with the number of pixels.

**Step 2, Thermal noise removal**
- The sensor adds a noise floor, the NESZ (noise-equivalent sigma zero). It is roughly −20 to −25 dB, depending on beam.
- Low-backscatter targets sit near this floor: calm water, smooth roads, and especially cross-pol HV over bare soil.
- Subtracting the noise level from the `calibration/` folder prevents false brightness and visible beam seams.

**Step 3, Radiometric calibration**
- This converts digital numbers (DN) into physically meaningful backscatter:

  **σ⁰ (or β⁰, γ⁰) = (DN² + B) / A**

  - *A* is the range-dependent gain from `lutSigma`, `lutBeta`, or `lutGamma`. It is interpolated across columns from `pixelFirstLutValue` and `stepSize`.
  - *B* is the offset.
- Which one to choose:
  - **β⁰** is radar brightness in slant geometry. It is the required input for terrain flattening.
  - **σ⁰** is normalized to the ellipsoid ground area. It is fine for flat terrain.
  - **γ⁰** is normalized to the area perpendicular to the beam. It is less incidence-angle dependent.
- For complex data (SLC), calibrate I and Q separately so the phase is preserved: divide by √A.

**Step 4, Multilooking (optional)**
- This averages pixels in range and azimuth to get roughly square ground pixels.
- It reduces speckle, at the cost of resolution.
- An example is averaging 2×2 on 16 m data to get a clean 30 m product.

**Step 5, Speckle filtering**
- Speckle is multiplicative noise inherent to coherent imaging.

| Filter | Character | When |
|---|---|---|
| Boxcar | Simple mean, blurs edges | Polarimetric matrices, large homogeneous areas |
| Lee / Refined Lee | Adaptive, preserves edges | General land mapping (default choice) |
| Gamma-MAP | Statistical model | Agriculture, forests |
| Lee Sigma | Preserves point targets | Urban, ships |
| Multi-temporal (Quegan) | Uses the stack over time | Time series; best detail retention |

- Use a 5×5 or 7×7 window.
- Do **not** filter before InSAR coregistration, because it destroys phase.
- Avoid heavy filtering before ship or point-target detection.

**Step 6, Radiometric terrain flattening**
- Slopes facing the sensor appear bright, and slopes facing away appear dark, purely from geometry.
- Terrain flattening (Small 2011) uses a DEM to compute the true illuminated area. It converts β⁰ into **terrain-flattened γ⁰ (γ⁰_T)**.
- It is essential in hilly or mountainous areas and for comparing ascending with descending passes.
- In flat agricultural plains it changes little.

**Step 7, Terrain correction (geocoding / orthorectification)**
- Range-Doppler terrain correction uses the orbit, timing, and a DEM to place each pixel at its true map location. This removes foreshortening displacement.
- Settings:
  - **DEM:** Copernicus GLO-30. SRTM 1-sec also works; use a local LiDAR DEM if available.
  - **Resampling:** bilinear for intensity. Nearest neighbour for masks or classes.
  - **Output pixel size:** match the beam mode (e.g., 30 m for 30M CP).
  - **Projection:** UTM or a national grid.
- Also export the **layover/shadow mask**, the **local incidence angle**, and the **projected local incidence angle**. They are critical for quality control and as model covariates.

**Step 8, Convert to dB**
- **dB = 10·log₁₀(σ⁰)**
- Do statistics (averaging, filtering) in **linear** units, and use dB only for display and for ML features. Averaging dB values biases the result.

**Step 9, Quality control**
- Mask pixels flagged as layover or shadow.
- Mask pixels near the noise floor.
- Mask image edges that have no-data artifacts.
- Check radiometric stability against known stable targets. Urban areas and dense forest should stay near-constant over time.


Compact-polarimetry (CP)
Compact-polarimetry (CP) is a SAR mode where the radar transmits a circularly polarized wave (commonly RH or LH) and receives two orthogonal linear polarizations (Horizontal, RH and Vertical, RV). This enables the construction of a 2×2 complex covariance matrix, capturing much of the information of full quad-pol but at the cost and coverage of dual-pol.


- The radar records complex scattering coefficients: **S<sub>RH</sub>** and **S<sub>RV</sub>** (where RH = Received Horizontal, RV = Received Vertical, both after transmitting Right-Circular polarization).

- The covariance matrix **[C<sub>2</sub>]** is formed as:

  ```
  C₂ = | <|S_RH|²>      <S_RH S_RV*>   |
       | <S_RV S_RH*>   <|S_RV|²>      |
  ```
  or, in markdown math:
  $$
  \mathbf{C}_2 =
  \begin{bmatrix}
    \langle |S_{\mathrm{RH}}|^2 \rangle & \langle S_{\mathrm{RH}} S_{\mathrm{RV}}^* \rangle \\
    \langle S_{\mathrm{RV}} S_{\mathrm{RH}}^* \rangle & \langle |S_{\mathrm{RV}}|^2 \rangle
  \end{bmatrix}
  $$
  where ⟨·⟩ indicates spatial averaging (speckle filtering).

- From this, you compute **Stokes parameters**:

  - $S_0 = \langle |S_{\mathrm{RH}}|^2 \rangle + \langle |S_{\mathrm{RV}}|^2 \rangle$ (total intensity)
  - $S_1 = \langle |S_{\mathrm{RH}}|^2 \rangle - \langle |S_{\mathrm{RV}}|^2 \rangle$
  - $S_2 = 2 \, \operatorname{Re}(\langle S_{\mathrm{RH}} S_{\mathrm{RV}}^* \rangle)$ (real part)
  - $S_3 = \pm 2 \, \operatorname{Im}(\langle S_{\mathrm{RH}} S_{\mathrm{RV}}^* \rangle)$ (imaginary part; sign depends on handedness convention)

- The **degree of polarization** is:

  $$
  m = \frac{\sqrt{S_1^2 + S_2^2 + S_3^2}}{S_0}
  $$
  This describes how "purely" the scene scatters polarization (low *m* = depolarized, e.g., vegetation).

- The **ellipticity angle** ($\chi$) (indicates even vs. odd bounce) is:

  $$
  \sin 2\chi = -\frac{S_3}{m S_0}
  $$
  Negative means odd-bounce dominant (e.g., double-bounce in urban or flooded areas).

- The **relative phase** ($\delta$) between RH and RV is:

  $$
  \delta = \arctan\left(\frac{S_3}{S_2}\right)
  $$

**m-χ decomposition:** This produces an RGB visualization to distinguish scattering types:
- **Red channel (Double-bounce):** $ \sqrt{m S_0 \frac{1-\sin 2\chi}{2}} $
- **Green channel (Volume):** $ \sqrt{S_0 (1-m)} $
- **Blue channel (Surface):** $ \sqrt{m S_0 \frac{1+\sin 2\chi}{2}} $

Thus, compact-pol SAR enables physically interpretable, information-rich products using rigorous polarimetric math.


### Compact-polarimetry (CP) processing

CP is what distinguishes RCM. It captures much of quad-pol's information at dual-pol cost and swath width.

**1. Covariance matrix.** From the calibrated complex RH and RV channels, form the 2×2 covariance matrix [C2]:
- ⟨|RH|²⟩, ⟨|RV|²⟩, ⟨RH·RV*⟩
- ⟨ ⟩ denotes spatial averaging, which is the polarimetric speckle filter.

**2. Stokes parameters**
- S₀ = ⟨|RH|²⟩ + ⟨|RV|²⟩ is the total power.
- S₁ = ⟨|RH|²⟩ − ⟨|RV|²⟩
- S₂ = 2 Re⟨RH·RV*⟩
- S₃ = ±2 Im⟨RH·RV*⟩. The sign depends on transmit handedness, so check the convention in your software.

**3. Derived parameters**
- **Degree of polarization:** m = √(S₁² + S₂² + S₃²) / S₀
  - Low m means depolarized, volume-like scattering, as in vegetation.
- **Ellipticity angle:** sin 2χ = −S₃ / (m·S₀)
  - Indicates odd versus even bounce.
- **Relative phase:** δ = arctan(S₃ / S₂)

**4. m-χ decomposition (Raney).** This gives an RGB composite:
- Double bounce = √[m·S₀·(1 − sin 2χ)/2], displayed in red. Indicates buildings and flooded vegetation.
- Volume = √[S₀·(1 − m)], displayed in green. Indicates forest and crop canopy.
- Surface = √[m·S₀·(1 + sin 2χ)/2], displayed in blue. Indicates bare soil and water.
- **m-δ decomposition** follows the same logic, using δ in place of χ.

**5. Other CP products**
- Circular polarization ratio (CPR) indicates surface roughness and is used for sea ice.
- RH/RV backscatter and the conformity coefficient are used for soil moisture.
- **Pseudo-quad reconstruction** (Souyris or Nord methods) estimates a full 3×3 matrix. This lets you apply quad-pol decompositions such as Freeman-Durden or H/A/α.

**6. Order of operations.** Terrain-correct the decomposition outputs *after* computing them, not before. Resampling complex data first corrupts the phase relationships.

**Applications:** crop type and phenology, wetland classes (open water vs. flooded vegetation vs. marsh), sea-ice typing, and oil-slick vs. look-alike discrimination.


### InSAR processing (SLC)

**1. Pair or stack selection**
- Use the same beam mode, orbit direction, and satellite geometry.
- Prefer short temporal baselines. RCM's 4-day repeat is a major advantage for coherence.
- Prefer small perpendicular baselines for deformation work.

**2. Coregistration**
- Use DEM-assisted coregistration to sub-pixel accuracy, about 0.1 pixel.

**3. Interferogram formation**
- Multiply one image by the complex conjugate of the other.
- Remove flat-earth phase.
- Estimate **coherence** (0 to 1), which measures phase reliability.

**4. Topographic phase removal**
- Subtract the DEM-simulated phase. What remains is **differential InSAR (DInSAR)**, containing deformation, atmosphere, and noise.

**5. Goldstein adaptive filtering**
- Reduces phase noise before unwrapping.

**6. Phase unwrapping**
- Use SNAPHU, exported from SNAP and re-imported.
- Wrapped phase cycles through −π to π. Unwrapping converts it into a continuous phase.

**7. Phase to displacement**
- **d = −(λ / 4π)·Δφ**
- With λ ≈ 5.55 cm, one fringe is about 2.8 cm of line-of-sight (LOS) motion.

**8. Geocode, then time-series analysis**
- Run SBAS or Persistent Scatterer analysis in MintPy, which reads SNAP outputs, or in StaMPS.
- These separate deformation from atmospheric noise.

**9. Decomposition**
- Combine ascending and descending LOS results to estimate vertical and east-west motion.

**Applications:** subsidence, landslides, infrastructure monitoring, permafrost, and coherence-based change or damage maps.


### Downstream analysis

| Task | Typical method |
|---|---|
| Flood mapping | Threshold low γ⁰ in VV/HH or CP (Otsu, split-based), plus a HAND (height above nearest drainage) mask |
| Change detection | Log-ratio or difference of dB images; coherence drop |
| Ship detection | CFAR (constant false alarm rate) on unfiltered HH or HV, land-masked |
| Crop classification | Multi-temporal γ⁰ + CP features + local incidence angle → Random Forest, XGBoost, or temporal deep learning |
| Soil moisture | Change detection on a time series, or a water-cloud model with a vegetation correction |
| Biomass / forest | Volume-component and cross-pol backscatter saturate early at C-band; combine with L-band |


### Tools

**ESA SNAP**
- Full GUI and the `gpt` command line.
- Native RCM reader, calibration, CP tools, InSAR, and terrain correction.
- Build the chain in the Graph Builder and batch it.

**Python**
- `esa_snappy` or `subprocess` + `gpt` for automation.
- `rasterio`, `xarray`, and `rioxarray` for arrays.
- `sarsen` for radiometric terrain correction (RTC).
- `scikit-image` for Otsu thresholding.
- `MintPy` for InSAR time series.

**Other options**
- **NRCan RCM CEOS-ARD:** already calibrated, noise-corrected, and terrain-corrected compact-pol at 30 m over Canada. You can skip steps 1 to 8 entirely.
- **Commercial alternatives:** PCI Catalyst, GAMMA, and SARscape.


### Common pitfalls

- Opening the TIFF instead of `product.xml`, which loses calibration and geometry.
- Averaging or filtering in dB instead of linear units.
- Terrain-correcting complex data before polarimetric or InSAR steps.
- Skipping terrain flattening in hilly terrain, then attributing slope effects to land cover.
- Mixing beam modes, incidence angles, or ascending and descending passes in one time series without normalization.
- Ignoring the noise floor in cross-pol channels over smooth surfaces.


## Additional Resources

- [CSA, RADARSAT satellites technical comparison](https://www.asc-csa.gc.ca/eng/satellites/radarsat/technical-features/radarsat-comparison.asp)
- [CSA, RCM technical characteristics](https://asc-csa.gc.ca/eng/satellites/radarsat/technical-features/characteristics.asp)
- [CATALYST, RADARSAT Constellation reference](https://catalyst.earth/catalyst-system-files/help/references/gdb_r/RADARSAT_Constellation.html)
- [CSA Open Data, RCM Data](https://data.asc-csa.gc.ca/en/dataset/64927856-8o0t-5u1o-987izna480aa45)
- [EODMS RCM CEOS-ARD (geo.ca)](https://app.geo.ca/en-ca/map-browser/record/eodms-rcm-ard)
- [AWS Open Data, RCM CEOS ARD](https://registry.opendata.aws/rcm-ceos-ard/)
- [RCM modes for sea ice (sciforum)](https://sciforum.net/paper/download/5136/manuscript)
- [CEOS LSI-VC, CSA presentation](https://ceos.org/document_management/Virtual_Constellations/LSI/Meetings/LSI-VC-14/Presentations/2.6_CSA_Yves.pdf)
- [Government of Canada, RCM replenishment satellite (June 2026)](https://www.canada.ca/en/space-agency/news/2026/06/minister-joly-announces-688m-to-strengthen-canadas-sovereign-satellite-capacity-with-radarsat-constellation-mission-replenishment-satellite.html)
- [The Canadian Press via CKOM](https://www.ckom.com/?p=1063597)
- [The Logic](https://thelogic.co/briefing/canada-launches-three-satellites-into-orbit-aboard-spacex-rocket/)


**Installation.** This notebook needs installation option 1 (vector, raster, CRS and visualization): `pip install "geolattice[core]"`. See "Installation options" in the GeoLattice README.